In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy import constants, units
from astropy.coordinates import Angle
from scipy.optimize import curve_fit
import pandas as pd
import os

In [2]:
rootdir = '/Users/thepoetoftwilight/Documents/CUBS/Data/PG1522+101/'

Load in the photometry catalogs - first MUSE

In [3]:
MUSE_df = pd.read_csv(rootdir+'MUSE/pseudo_gri_photometry_final.dat')

In [4]:
MUSE_df

,ID,RA,Dec,pseudo_g_mag,pseudo_r_mag,pseudo_i_mag,z
0,1,231.107352,9.967345,23.22,21.99,21.41,0.5356
1,2,231.105119,9.967082,23.43,22.76,22.62,0.4581
2,3,231.096160,9.966556,25.12,24.19,23.21,0.9576
3,5,231.104043,9.966569,25.63,24.67,24.01,0.8217
4,18,231.103734,9.981672,19.83,19.34,19.14,0.0000
...,...,...,...,...,...,...,...
75,109,231.109193,9.967189,27.75,27.10,26.66,-1.0000
76,114,231.106305,9.982892,24.74,23.53,22.84,0.0000
77,115,231.108073,9.982911,25.49,25.19,24.46,-1.0000
78,122,231.101596,9.982185,25.21,24.75,24.70,0.0966


Next, HST 160W

In [5]:
F160W_df = pd.read_csv(rootdir+'HST_images/f160w_photometry_final.dat')

In [6]:
F160W_df

,ID,RA,Dec,f160w_mag
0,1,231.089271,9.957233,19.43
1,2,231.090045,9.956782,20.07
2,3,231.090317,9.953885,21.75
3,4,231.091208,9.952799,23.04
4,5,231.088950,9.954643,18.60
...,...,...,...,...
1077,1078,231.097474,9.988867,23.78
1078,1079,231.103001,9.990440,23.31
1079,1080,231.109894,9.992215,21.65
1080,1081,231.100346,9.989033,20.26


Next, HST 140W

In [7]:
F140W_df = pd.read_csv(rootdir+'HST_images/f140w_photometry_final.dat')

In [8]:
F140W_df

,ID,RA,Dec,f140w_mag
0,1,231.087323,9.960302,19.63
1,2,231.087641,9.959497,18.65
2,3,231.117408,9.965167,19.66
3,4,231.118858,9.964090,18.02
4,5,231.091122,9.953285,22.92
...,...,...,...,...
1109,1110,231.092249,9.987659,22.68
1110,1111,231.115917,9.960264,23.78
1111,1112,231.099336,9.989730,22.30
1112,1113,231.087405,9.986115,23.89


Finally, HST 814W

In [9]:
F814W_df = pd.read_csv(rootdir+'HST_images/f814w_photometry_final.dat')

In [10]:
F814W_df

,ID,RA,Dec,f814w_mag
0,1,231.104821,10.013471,18.49
1,2,231.083832,10.009046,16.76
2,3,231.094990,10.015184,21.75
3,4,231.097115,10.017805,18.76
4,5,231.099848,10.019035,21.58
...,...,...,...,...
8043,8044,231.121165,9.969147,25.39
8044,8045,231.124451,9.971414,26.28
8045,8046,231.083398,9.957430,28.42
8046,8047,231.117794,9.968139,28.36


We'll compute the angular separation between all possible pairs of objects using the formula. Below, we present the formalism -

Point 1: ($\alpha_1, \delta_1$), Point 2: ($\alpha_2, \delta_2$)

Angular separation $\phi$ between unit vectors -

$$\cos(\phi) = \hat{r_1} \cdot \hat{r_2}$$

$$\Rightarrow \cos(\phi) = \cos(\delta_1) \cos(\delta_2) \cos(\alpha_1) \cos(\alpha_2) + \cos(\delta_1) \cos(\delta_2) \sin(\alpha_1) \sin(\alpha_2) + \sin(\delta_1) \sin(\delta_2)$$

$$\boxed{\phi = \arccos[ \cos(\delta_1) \cos(\delta_2) \cos(\alpha_1) \cos(\alpha_2) + \cos(\delta_1) \cos(\delta_2) \sin(\alpha_1) \sin(\alpha_2) + \sin(\delta_1) \sin(\delta_2)]}$$

In [11]:
def calc_phi(alpha_1, delta_1, alpha_2, delta_2):
    
    return np.arccos(np.dot([np.cos(delta_1)*np.cos(alpha_1), np.cos(delta_1)*np.sin(alpha_1), np.sin(delta_1)],
                      [np.cos(delta_2)*np.cos(alpha_2), np.cos(delta_2)*np.sin(alpha_2), np.sin(delta_2)]))

We'll define a function to create a grid of angular separations

In [12]:
def calc_phi_grid(df_1, df_2):
    
    # Get RAs and Decs from each dataframe
    df_1_RA = np.array(df_1['RA']*np.pi/180)
    df_1_Dec = np.array(df_1['Dec']*np.pi/180)
    
    df_2_RA = np.array(df_2['RA']*np.pi/180)
    df_2_Dec = np.array(df_2['Dec']*np.pi/180)
    
    # Define a grid of phi values
    phi_grid = np.zeros((len(df_1), len(df_2)))
    
    for i in range(len(df_1_RA)):
        
        phi_grid[i,:] = (calc_phi(df_1_RA[i], df_1_Dec[i], 
                                  df_2_RA, df_2_Dec)*units.radian).to(units.arcsecond).value
  
    return phi_grid

In [13]:
def calc_match_indices(phi_grid, phi_thresh=1):
    
    # Record the indices of closely separated objects, and also their separation
    match_idx_1 = []
    match_idx_2 = []
    match_phi = []
    
    for i in range(phi_grid.shape[0]):
        
        phi_slice = phi_grid[i,:]
        
        if np.min(phi_slice)<=phi_thresh:
            match_idx_1.append(i)
            match_idx_2.append(np.argmin(phi_slice))
            match_phi.append(np.min(phi_slice))
                
    return np.array(match_idx_1), np.array(match_idx_2), np.array(match_phi)

First, match F140W to F160W

In [14]:
phi_grid_HST = calc_phi_grid(F140W_df, F160W_df)

In [15]:
# Get the matching indices for the two HST catalogs
F140W_df_idx, F160W_df_idx, phi_match_HST = calc_match_indices(phi_grid_HST, phi_thresh=0.5)

In [16]:
len(F140W_df), len(F140W_df_idx)

(1114, 686)

In [17]:
# Now create a copy of the F140W catalog
master_df = F140W_df.copy()

In [18]:
master_df['f160w_mag'] = ['-1.0' for i in range(len(master_df))]
master_df['phi_HST'] = ['-1.0' for i in range(len(master_df))]

In [19]:
master_df.loc[F140W_df_idx, 'f160w_mag'] = np.array(F160W_df.loc[F160W_df_idx,'f160w_mag'])
master_df.loc[F140W_df_idx, 'phi_HST'] = phi_match_HST

In [20]:
master_df.loc[F140W_df_idx]

,ID,RA,Dec,f140w_mag,f160w_mag,phi_HST
5,6,231.091197,9.952817,21.23,23.04,0.07684
6,7,231.090552,9.954074,20.90,21.52,0.145793
7,8,231.090286,9.953914,20.46,21.75,0.154629
15,16,231.122133,9.964281,20.15,20.22,0.25483
16,17,231.092749,9.955428,22.08,22.76,0.360357
...,...,...,...,...,...,...
1106,1107,231.100728,9.990449,23.34,23.25,0.090188
1108,1109,231.086151,9.985939,23.74,23.64,0.141552
1109,1110,231.092249,9.987659,22.68,22.17,0.102309
1110,1111,231.115917,9.960264,23.78,23.49,0.061778


Now, merge in the MUSE catalog

In [21]:
phi_grid_MUSE = calc_phi_grid(master_df, MUSE_df)

In [22]:
master_df_idx, MUSE_df_idx, phi_match_MUSE = calc_match_indices(phi_grid_MUSE, phi_thresh=0.5)

In [23]:
master_df_new = master_df.copy()

In [24]:
master_df_new['pseudo_g_mag'] = ['-1.0' for i in range(len(master_df_new))]
master_df_new['pseudo_r_mag'] = ['-1.0' for i in range(len(master_df_new))]
master_df_new['pseudo_i_mag'] = ['-1.0' for i in range(len(master_df_new))]
master_df_new['phi_MUSE'] = ['-1.0' for i in range(len(master_df_new))]
master_df_new['z'] = ['-1.0' for i in range(len(master_df_new))]

In [25]:
master_df_new.loc[master_df_idx, 'pseudo_g_mag'] = np.array(MUSE_df.loc[MUSE_df_idx, 'pseudo_g_mag'])
master_df_new.loc[master_df_idx, 'pseudo_r_mag'] = np.array(MUSE_df.loc[MUSE_df_idx, 'pseudo_r_mag'])
master_df_new.loc[master_df_idx, 'pseudo_i_mag'] = np.array(MUSE_df.loc[MUSE_df_idx, 'pseudo_i_mag'])
master_df_new.loc[master_df_idx, 'phi_MUSE'] = phi_match_MUSE
master_df_new.loc[master_df_idx, 'z'] = np.array(MUSE_df.loc[MUSE_df_idx, 'z'])

In [26]:
len(master_df_idx)

79

In [27]:
master_df_new.loc[master_df_idx]

,ID,RA,Dec,f140w_mag,f160w_mag,phi_HST,pseudo_g_mag,pseudo_r_mag,pseudo_i_mag,phi_MUSE,z
210,211,231.098500,9.983185,20.60,20.67,0.064545,24.48,23.64,23.08,0.127174,0.4784
213,214,231.097411,9.982920,21.42,21.41,0.068727,24.54,24.2,23.8,0.221874,0.1393
224,225,231.094330,9.981459,21.42,21.4,0.018947,25.75,25.44,25.28,0.227258,-1.0
228,229,231.097686,9.982070,21.17,21.27,0.064105,26.15,25.71,25.25,0.162232,-1.0
229,230,231.097092,9.982013,20.82,20.65,0.038756,27.11,26.62,25.78,0.260113,-1.0
...,...,...,...,...,...,...,...,...,...,...,...
760,761,231.106633,9.969961,19.84,19.8,0.039361,28.88,27.78,26.77,0.470329,-1.0
780,781,231.105125,9.967151,20.40,20.5,0.05575,23.43,22.76,22.62,0.249112,0.4581
784,785,231.104042,9.966671,20.99,21.0,0.065056,25.63,24.67,24.01,0.369534,0.8217
786,787,231.107361,9.967401,19.13,18.99,0.058236,23.22,21.99,21.41,0.202414,0.5356


In [28]:
#master_df_new = master_df_new.loc[master_df_idx]
#master_df_new['ID'] = np.arange(1, len(master_df_new)+1)
#master_df_new.reset_index(drop=True, inplace=True)

Finally, match to F814W

In [29]:
phi_grid_814W = calc_phi_grid(master_df_new, F814W_df)

In [30]:
# Get the matching indices for the two HST catalogs
master_df_new_idx, F814W_df_idx, phi_match_814W = calc_match_indices(phi_grid_814W, phi_thresh=0.5)

In [31]:
len(master_df_new), len(master_df_new_idx)

(1114, 645)

In [32]:
# Now create a copy of the F140W catalog
master_df_final = master_df_new.copy()

In [33]:
master_df_final['f814w_mag'] = ['-1.0' for i in range(len(master_df_final))]

In [34]:
master_df_final.loc[master_df_new_idx, 'f814w_mag'] = np.array(F814W_df.loc[F814W_df_idx,'f814w_mag'])

Here is the final catalog

In [35]:
master_df_final.loc[master_df_new_idx]

,ID,RA,Dec,f140w_mag,f160w_mag,phi_HST,pseudo_g_mag,pseudo_r_mag,pseudo_i_mag,phi_MUSE,z,f814w_mag
2,3,231.117408,9.965167,19.66,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,26.51
46,47,231.102018,9.990466,23.81,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,28.68
47,48,231.083504,9.974991,21.29,22.68,0.033388,-1.0,-1.0,-1.0,-1.0,-1.0,27.65
48,49,231.081069,9.977999,23.00,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,25.61
51,52,231.082748,9.974601,21.25,21.88,0.02195,-1.0,-1.0,-1.0,-1.0,-1.0,25.99
...,...,...,...,...,...,...,...,...,...,...,...,...
1106,1107,231.100728,9.990449,23.34,23.25,0.090188,-1.0,-1.0,-1.0,-1.0,-1.0,27.73
1108,1109,231.086151,9.985939,23.74,23.64,0.141552,-1.0,-1.0,-1.0,-1.0,-1.0,27.61
1109,1110,231.092249,9.987659,22.68,22.17,0.102309,-1.0,-1.0,-1.0,-1.0,-1.0,26.89
1111,1112,231.099336,9.989730,22.30,22.56,0.151232,-1.0,-1.0,-1.0,-1.0,-1.0,26.09


Compile the final catalog

In [36]:
gal_ra_arr = np.array(master_df_final['RA'])
gal_dec_arr = np.array(master_df_final['Dec'])
gal_f160w_mag_arr = np.array(master_df_final['f160w_mag'])
gal_f140w_mag_arr = np.array(master_df_final['f140w_mag'])
gal_f814w_mag_arr = np.array(master_df_final['f814w_mag'])
gal_pseudo_g_mag_arr = np.array(master_df_final['pseudo_g_mag'])
gal_pseudo_r_mag_arr = np.array(master_df_final['pseudo_r_mag'])
gal_pseudo_i_mag_arr = np.array(master_df_final['pseudo_i_mag'])
gal_z_arr = np.array(master_df_final['z'])

In [37]:
with open(rootdir+'ldss_photometry_final.dat', 'w') as f:

    f.write('RA,Dec,f160w_mag,f140w_mag,f814w_mag,pseudo_g_mag,pseudo_r_mag,pseudo_i_mag,z')

    for i in range(len(gal_ra_arr)):

        f.write('\n'+str(gal_ra_arr[i])+','+
             str(gal_dec_arr[i])+','+
             str(gal_f160w_mag_arr[i])+','+
             str(gal_f140w_mag_arr[i])+','+
             str(gal_f814w_mag_arr[i])+','+
             str(gal_pseudo_g_mag_arr[i])+','+
             str(gal_pseudo_r_mag_arr[i])+','+
             str(gal_pseudo_i_mag_arr[i])+','+
             str(gal_z_arr[i]))

Also write the subset of MUSE objects into a separate catalog

In [38]:
master_df_overlap = master_df_final.loc[master_df_new_idx]

In [39]:
len(master_df_overlap)

645

In [40]:
gal_ra_overlap_arr = np.array(master_df_overlap['RA'])
gal_dec_overlap_arr = np.array(master_df_overlap['Dec'])
gal_f160w_mag_overlap_arr = np.array(master_df_overlap['f160w_mag'])
gal_f140w_mag_overlap_arr = np.array(master_df_overlap['f140w_mag'])
gal_f814w_mag_overlap_arr = np.array(master_df_overlap['f814w_mag'])
gal_pseudo_g_mag_overlap_arr = np.array(master_df_overlap['pseudo_g_mag'])
gal_pseudo_r_mag_overlap_arr = np.array(master_df_overlap['pseudo_r_mag'])
gal_pseudo_i_mag_overlap_arr = np.array(master_df_overlap['pseudo_i_mag'])
gal_z_overlap_arr = np.array(master_df_overlap['z'])

In [41]:
with open(rootdir+'ldss_photometry_final_subset.dat', 'w') as f:

    f.write('RA,Dec,f160w_mag,f140w_mag,f814w_mag,pseudo_g_mag,pseudo_r_mag,pseudo_i_mag,z')

    for i in range(len(gal_ra_overlap_arr)):

        f.write('\n'+str(gal_ra_overlap_arr[i])+','+
             str(gal_dec_overlap_arr[i])+','+
             str(gal_f160w_mag_overlap_arr[i])+','+
             str(gal_f140w_mag_overlap_arr[i])+','+
             str(gal_f814w_mag_overlap_arr[i])+','+
             str(gal_pseudo_g_mag_overlap_arr[i])+','+
             str(gal_pseudo_r_mag_overlap_arr[i])+','+
             str(gal_pseudo_i_mag_overlap_arr[i])+','+
             str(gal_z_overlap_arr[i]))